# Creating JSON file

In [1]:
# Install the COCO API
!pip install pycocotools

# # Essential libraries for your multimodal task
# import torch
# import clip # If you are using OpenAI's CLIP
# import requests
# from PIL import Image
# from io import BytesIO

In [2]:
# This downloads only the annotation zip (~241MB compressed)
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip annotations_trainval2017.zip

--2026-04-06 13:27:48--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.183.207, 52.216.42.153, 16.182.66.73, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.183.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘annotations_trainval2017.zip’

annotations_trainva 100%[===================>] 241.19M  76.7MB/s    in 3.3s    

2026-04-06 13:27:52 (73.1 MB/s) - ‘annotations_trainval2017.zip’ saved [252907541/252907541]

Archive:  annotations_trainval2017.zip
  inflating: annotations/instances_train2017.json  
  inflating: annotations/instances_val2017.json  
  inflating: annotations/captions_train2017.json  
  inflating: annotations/captions_val2017.json  
  inflating: annotations/person_keypoints_train2017.json  
  inflating: annotations/person_keypoints_val2017.json  


In [3]:
import os
print(f"Instances size: {os.path.getsize('annotations/instances_train2017.json') / (1024*1024):.2f} MB")
# Should show ~448 MB

Instances size: 448.02 MB


In [4]:
import json
import random
from pycocotools.coco import COCO

# 1. Initialize COCO API
# Set paths to the unzipped files
annFile = './annotations/instances_train2017.json'
capFile = './annotations/captions_train2017.json'

try:
    print("Loading COCO Annotations...")
    coco = COCO(annFile) # Object detection labels
    coco_caps = COCO(capFile) # Text captions
    print("Success!")
except Exception as e:
    print(f"Error: {e}. Try re-downloading the annotation zip.")

# Continue with your 1K per class + 2K random sampling logic

target_classes = ['bear', 'zebra', 'elephant', 'giraffe', 'horse', 'dog', 'cat', 'airplane', 'train', 'person']
N_CHOICES = 12

seen_img_ids = set()
class_to_imgs = {cls: [] for cls in target_classes}

Loading COCO Annotations...
loading annotations into memory...
Done (t=35.16s)
creating index...
index created!
loading annotations into memory...
Done (t=1.94s)
creating index...
index created!
Success!


In [5]:
# ---------------------------------------------------------
# PHASE 1: Collect 1,000 exclusive images per class
# ---------------------------------------------------------
print("\nSampling 1K exclusive images per class...")
for cls_name in target_classes:
    catIds = coco.getCatIds(catNms=[cls_name])
    imgIds = coco.getImgIds(catIds=catIds)

    # Shuffle to ensure random selection and avoid time/video bias
    random.shuffle(imgIds)

    count = 0
    for img_id in imgIds:
        if img_id not in seen_img_ids:
            class_to_imgs[cls_name].append(img_id)
            seen_img_ids.add(img_id)
            count += 1
        if count >= 1000:
            break

    print(f"  - {cls_name}: {count} images")


Sampling 1K exclusive images per class...
  - bear: 960 images
  - zebra: 1000 images
  - elephant: 1000 images
  - giraffe: 1000 images
  - horse: 1000 images
  - dog: 1000 images
  - cat: 1000 images
  - airplane: 1000 images
  - train: 1000 images
  - person: 1000 images


In [6]:
# ---------------------------------------------------------
# PHASE 2: Collect 2,000 random images from the same classes
# ---------------------------------------------------------
print("\nSampling 2K random additional images...")
random_pool = []
for cls_name in target_classes:
    catIds = coco.getCatIds(catNms=[cls_name])
    imgIds = coco.getImgIds(catIds=catIds)
    for img_id in imgIds:
        if img_id not in seen_img_ids:
            random_pool.append((img_id, cls_name))

random.shuffle(random_pool)
random_2k = random_pool[:2000]

for img_id, cls_name in random_2k:
    class_to_imgs[cls_name].append(img_id)
    seen_img_ids.add(img_id)

print(f"Total images collected: {len(seen_img_ids)}")


Sampling 2K random additional images...
Total images collected: 11956


In [7]:
# ---------------------------------------------------------
# PHASE 3: Build the N-Way Classification Dataset
# ---------------------------------------------------------
print("\nBuilding N-way prompt structures...")

# First, gather a pool of all captions for each class to use as distractors
class_to_captions = {cls: [] for cls in target_classes}
for cls_name, img_ids in class_to_imgs.items():
    for img_id in img_ids:
        annIds = coco_caps.getAnnIds(imgIds=img_id)
        anns = coco_caps.loadAnns(annIds)
        class_to_captions[cls_name].extend([ann['caption'] for ann in anns])

final_dataset = []

for cls_name, img_ids in class_to_imgs.items():
    for img_id in img_ids:
        # 1. Get the Correct Caption
        annIds = coco_caps.getAnnIds(imgIds=img_id)
        anns = coco_caps.loadAnns(annIds)
        if not anns:
            continue # Skip if no caption exists

        current_img_captions = [ann['caption'] for ann in anns]
        correct_caption = random.choice(current_img_captions)

        # 2. Select Distractor Captions from both SAME class and OTHER classes
        same_class_pool = [cap for cap in class_to_captions[cls_name] if cap not in current_img_captions]
        other_classes = [c for c in target_classes if c != cls_name]

        # Determine the mix: e.g., 2 from the same class, remaining from others
        num_same_class_distractors = 2
        num_other_distractors = (N_CHOICES - 1) - num_same_class_distractors

        # Safely get distractors from the same class
        same_class_distractors = random.sample(same_class_pool, min(num_same_class_distractors, len(same_class_pool)))

        # Compensate if there weren't enough same-class distractors available
        num_other_distractors += (num_same_class_distractors - len(same_class_distractors))

        # Get distractors from other classes
        distractor_classes = random.sample(other_classes, num_other_distractors)
        other_distractors = [random.choice(class_to_captions[d_class]) for d_class in distractor_classes]

        # Combine all valid distractors
        distractors = same_class_distractors + other_distractors

        # 4. Combine and Shuffle
        candidates = [correct_caption] + distractors
        random.shuffle(candidates)

        # 5. Track the Correct Index!
        ground_truth_idx = candidates.index(correct_caption)

        # 6. Fetch Image URL for Colab memory savings
        img_info = coco.loadImgs(img_id)[0]

        sample = {
            "image_id": img_id,
            "image_url": img_info['coco_url'],
            "true_label": cls_name,
            "choices": candidates,
            "ground_truth_idx": ground_truth_idx
        }
        final_dataset.append(sample)

print(f"\nSuccessfully created {len(final_dataset)} multimodal task samples!")
print("Sample structure:")
print(final_dataset[0])


Building N-way prompt structures...

Successfully created 11960 multimodal task samples!
Sample structure:
{'image_id': 439940, 'image_url': 'http://images.cocodataset.org/train2017/000000439940.jpg', 'true_label': 'bear', 'choices': ['A train on a railroad track with another rack running along side.', 'two jets flying in the air near one another', 'A dog is standing in front of a ball on the ground', 'A couple of giraffe standing in a field by a tree.', 'black bear standing on top of rock with rocky background', 'A policeman riding a horse in a city.', 'A person sits on the floor leaning against a bed. ', 'A cat who is perched on a bathroom sink.', 'Three zebras standing on grass in front of trees and bushes.', 'An Asian black bear sleeps in a woven hammock. ', 'Four elephants partake in shade underneath a tree.', 'two brown bears lying together and relaxing on a rock'], 'ground_truth_idx': 9}


In [8]:
with open('coco_multimodal_subset.json', 'w') as f:
    json.dump(final_dataset, f)

from google.colab import files
files.download('coco_multimodal_subset.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Downloading files

In [9]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from PIL import Image
from io import BytesIO

# 1. Create a directory for the images
IMAGE_DIR = '/content/coco_subset_images'
os.makedirs(IMAGE_DIR, exist_ok=True)

def download_image(sample):
    img_id = sample['image_id']
    url = sample['image_url']
    save_path = os.path.join(IMAGE_DIR, f"{img_id}.jpg")

    # Skip if already downloaded (useful if the script crashes)
    if os.path.exists(save_path):
        return

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            # We open and save via PIL to ensure the image isn't corrupted
            img = Image.open(BytesIO(response.content)).convert("RGB")
            img.save(save_path)
    except Exception as e:
        print(f"Failed to download {img_id}: {e}")

# 2. Run the download in parallel (Fast!)
print(f"Downloading {len(final_dataset)} images to {IMAGE_DIR}...")
with ThreadPoolExecutor(max_workers=20) as executor:
    list(tqdm(executor.map(download_image, final_dataset), total=len(final_dataset)))

print("\nDownload Complete!")

100%|██████████| 11960/11960 [03:40<00:00, 54.21it/s]


Download Complete!


In [10]:
# Add local path to each sample
for sample in final_dataset:
    sample['local_path'] = os.path.join(IMAGE_DIR, f"{sample['image_id']}.jpg")

# Re-save the JSON so you have the paths ready for your DataLoader
with open('coco_multimodal_subset_local.json', 'w') as f:
    json.dump(final_dataset, f)

In [11]:
final_dataset[0]

{'image_id': 439940,
 'image_url': 'http://images.cocodataset.org/train2017/000000439940.jpg',
 'true_label': 'bear',
 'choices': ['A train on a railroad track with another rack running along side.',
  'two jets flying in the air near one another',
  'A dog is standing in front of a ball on the ground',
  'A couple of giraffe standing in a field by a tree.',
  'black bear standing on top of rock with rocky background',
  'A policeman riding a horse in a city.',
  'A person sits on the floor leaning against a bed. ',
  'A cat who is perched on a bathroom sink.',
  'Three zebras standing on grass in front of trees and bushes.',
  'An Asian black bear sleeps in a woven hammock. ',
  'Four elephants partake in shade underneath a tree.',
  'two brown bears lying together and relaxing on a rock'],
 'ground_truth_idx': 9,
 'local_path': '/content/coco_subset_images/439940.jpg'}

In [12]:
!zip -r coco_images.zip /content/coco_subset_images

Streaming output truncated to the last 5000 lines.
  adding: content/coco_subset_images/295693.jpg (deflated 0%)
  adding: content/coco_subset_images/428004.jpg (deflated 1%)
  adding: content/coco_subset_images/290812.jpg (deflated 1%)
  adding: content/coco_subset_images/79196.jpg (deflated 1%)
  adding: content/coco_subset_images/368876.jpg (deflated 0%)
  adding: content/coco_subset_images/563155.jpg (deflated 1%)
  adding: content/coco_subset_images/86031.jpg (deflated 0%)
  adding: content/coco_subset_images/415530.jpg (deflated 0%)
  adding: content/coco_subset_images/129166.jpg (deflated 1%)
  adding: content/coco_subset_images/185432.jpg (deflated 0%)
  adding: content/coco_subset_images/25566.jpg (deflated 0%)
  adding: content/coco_subset_images/77432.jpg (deflated 1%)
  adding: content/coco_subset_images/538586.jpg (deflated 4%)
  adding: content/coco_subset_images/283757.jpg (deflated 0%)
  adding: content/coco_subset_images/33901.jpg (deflated 0%)
  adding: content/coco_s